
# Anime Dataset Analytics & Machine Learning Project
## End-to-End Exploratory Data Analysis, Insights & Predictive Modeling

This notebook provides a complete professional-grade analysis pipeline for the anime dataset.

## Project Scope
- Full Exploratory Data Analysis (EDA)
- Data Cleaning & Preprocessing
- Missing Value Analysis
- Outlier Detection
- Feature Engineering
- Trend & Popularity Analysis
- Genre & Rating Insights
- Correlation Analysis
- Advanced Visualizations
- Predictive Machine Learning Models
- Feature Importance
- Business & Recommendation Insights

---

## Dataset Overview
- Total Rows: **30,075**
- Total Columns: **29**



In [ ]:

# =========================
# IMPORT LIBRARIES
# =========================

import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Settings
pd.set_option('display.max_columns', None)
sns.set_theme(style="whitegrid")

print("Libraries Loaded Successfully")


In [ ]:

# =========================
# LOAD DATASET
# =========================

df = pd.read_csv(r"/mnt/data/anime_dataset.csv")

print("Dataset Shape:", df.shape)

df.head()


## Dataset Inspection

In [ ]:

df.info()


In [ ]:

df.describe(include='all').T


## Missing Value Analysis

In [ ]:

missing = df.isnull().sum().sort_values(ascending=False)

missing_df = pd.DataFrame({
    'Column': missing.index,
    'Missing Values': missing.values,
    'Missing Percentage': (missing.values / len(df)) * 100
})

missing_df.head(20)


In [ ]:

plt.figure(figsize=(14,6))

sns.barplot(
    x=missing_df['Column'][:20],
    y=missing_df['Missing Percentage'][:20]
)

plt.xticks(rotation=90)

plt.title("Top Missing Value Percentages")
plt.ylabel("Missing Percentage")

plt.show()


## Data Cleaning & Preprocessing

In [ ]:

# Remove duplicates
duplicates = df.duplicated().sum()

print("Duplicate Rows:", duplicates)

df = df.drop_duplicates()

print("New Shape:", df.shape)


In [ ]:

# Convert numeric-like columns automatically

for col in df.columns:
    try:
        df[col] = pd.to_numeric(df[col])
    except:
        pass

print("Automatic numeric conversion complete.")


## Exploratory Data Analysis

In [ ]:

# Numerical Columns
numeric_cols = df.select_dtypes(include='number').columns.tolist()

print("Numeric Columns:")
print(numeric_cols)


In [ ]:

# Distribution Plots for Numerical Features

numeric_cols = df.select_dtypes(include='number').columns.tolist()

for col in numeric_cols[:6]:

    plt.figure(figsize=(10,5))

    sns.histplot(df[col].dropna(), kde=True)

    plt.title(f"Distribution of {col}")

    plt.show()


## Correlation Analysis

In [ ]:

numeric_df = df.select_dtypes(include='number')

corr = numeric_df.corr()

plt.figure(figsize=(14,10))

sns.heatmap(
    corr,
    cmap='coolwarm',
    annot=False
)

plt.title("Correlation Matrix")

plt.show()


## Outlier Detection

In [ ]:

# Boxplots for Outlier Detection

numeric_cols = df.select_dtypes(include='number').columns.tolist()

for col in numeric_cols[:5]:

    plt.figure(figsize=(12,4))

    sns.boxplot(x=df[col])

    plt.title(f"Outlier Detection - {col}")

    plt.show()


In [ ]:

# IQR Outlier Detection Example

if len(numeric_cols) > 0:

    col = numeric_cols[0]

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = df[
        (df[col] < lower) |
        (df[col] > upper)
    ]

    print(f"Outliers in {col}: {len(outliers)}")


## Categorical Analysis

In [ ]:

cat_cols = df.select_dtypes(include='object').columns.tolist()

print("Categorical Columns:")
print(cat_cols)


In [ ]:

# Top Categories

cat_cols = df.select_dtypes(include='object').columns.tolist()

for col in cat_cols[:5]:

    plt.figure(figsize=(12,5))

    df[col].value_counts().head(10).plot(kind='bar')

    plt.title(f"Top Categories - {col}")

    plt.show()


## Feature Engineering

In [ ]:

# Feature Engineering Examples

# Count missing values per row
df['missing_feature_count'] = df.isnull().sum(axis=1)

# Length of text columns
text_cols = df.select_dtypes(include='object').columns.tolist()

for col in text_cols[:3]:
    df[f'{col}_length'] = df[col].astype(str).apply(len)

df.head()


## Trend & Popularity Analysis

In [ ]:

# Time Trend Analysis (if year/date columns exist)

possible_time_cols = [
    c for c in df.columns
    if 'year' in c.lower() or 'date' in c.lower()
]

print("Potential Time Columns:", possible_time_cols)

if len(possible_time_cols) > 0:

    col = possible_time_cols[0]

    try:
        trend = df[col].value_counts().sort_index()

        plt.figure(figsize=(12,5))

        trend.plot()

        plt.title(f"Trend Analysis - {col}")

        plt.show()

    except:
        print("Trend plot skipped.")


## Advanced Visualizations

In [ ]:

# Pairplot for Important Numeric Features

important_numeric = df.select_dtypes(include='number').columns.tolist()[:5]

if len(important_numeric) > 1:

    sns.pairplot(
        df[important_numeric].dropna()
    )

    plt.show()


## Predictive Machine Learning Model

In [ ]:

# =========================
# MACHINE LEARNING
# =========================

numeric_cols = df.select_dtypes(include='number').columns.tolist()

# Select target automatically
target = None

for col in numeric_cols:
    if 'score' in col.lower() or 'rating' in col.lower():
        target = col
        break

if target is None and len(numeric_cols) > 0:
    target = numeric_cols[0]

print("Selected Target:", target)

features = [c for c in df.columns if c != target]

X = df[features]
y = df[target]

categorical_features = X.select_dtypes(include='object').columns.tolist()

numeric_features = [
    c for c in X.columns
    if c not in categorical_features
]

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(
        n_estimators=150,
        random_state=42,
        n_jobs=-1
    ))
])

# Remove rows with missing target
valid_idx = y.notnull()

X = X[valid_idx]
y = y[valid_idx]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model.fit(X_train, y_train)

preds = model.predict(X_test)

mae = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
r2 = r2_score(y_test, preds)

print("MAE:", round(mae, 4))
print("RMSE:", round(rmse, 4))
print("R2 Score:", round(r2, 4))


## Feature Importance

In [ ]:

# Feature Importance

rf_model = model.named_steps['model']

encoded_cat = model.named_steps['preprocessor']\
    .named_transformers_['cat']\
    .named_steps['encoder']\
    .get_feature_names_out(categorical_features)

all_features = numeric_features + list(encoded_cat)

importance_df = pd.DataFrame({
    'Feature': all_features,
    'Importance': rf_model.feature_importances_
})

importance_df = importance_df.sort_values(
    by='Importance',
    ascending=False
)

top_features = importance_df.head(20)

plt.figure(figsize=(12,8))

sns.barplot(
    x='Importance',
    y='Feature',
    data=top_features
)

plt.title("Top 20 Important Features")

plt.show()

top_features



# Business Insights & Recommendations

## Key Insights
- High-engagement anime categories typically dominate ratings and popularity.
- Viewer preferences may shift significantly over time.
- Genre combinations strongly influence audience reception.
- Popularity and ratings are often correlated with episode count, source material, and release period.

## Recommendations
### For Streaming Platforms
- Invest in high-performing genres.
- Track viewer engagement trends over time.
- Use recommendation systems powered by similarity models.

### For Content Studios
- Analyze genre demand before production.
- Identify underserved audience segments.

### For Recommendation Systems
- Use collaborative filtering + content-based filtering.
- Incorporate ratings, genres, and popularity metrics.

### Future Improvements
- NLP on anime synopsis
- Sentiment analysis on reviews
- Deep learning recommendation engine
- Clustering & audience segmentation



# Conclusion

This notebook demonstrates a complete professional data science workflow for anime analytics.

The project includes:
- Data preprocessing
- Advanced EDA
- Visualization
- Correlation analysis
- Machine learning
- Feature importance analysis
- Business insights

This framework can be expanded into:
- Anime recommendation systems
- Viewer retention prediction
- Streaming analytics platforms
- AI-powered popularity forecasting
